# 🚀 TRAINING RADAR PULSE DEINTERLEAVING - SERVER LOCAL 2x RTX 2080

**Paper:** [Radar Pulse Deinterleaving with Transformer](https://arxiv.org/abs/2503.13476)

---

## ⚠️ TRƯỚC KHI BẮT ĐẦU:

1. **Giải nén data** trước (BƯỚC 1)
2. **Chạy từng cell theo thứ tự** (Shift+Enter)
3. **Thời gian:** Setup ~5 phút, Training ~3-8 giờ

---

## 🔧 **BƯỚC 1: Giải nén Data**

In [ ]:
import subprocess
from pathlib import Path
import os

# ============== THAY ĐỔI ĐÂY ==============
REPO_DIR = Path("/home/Trang55/deinterleaving")   # ← thư mục chứa repo
# ==========================================

# Tìm file zip trong repo
zip_files = list(REPO_DIR.glob("*.zip"))
data_dir  = REPO_DIR / "data"

if data_dir.exists() and any(data_dir.glob("train/*.h5")):
    print("✅ Data đã giải nén rồi! Bỏ qua bước này.")
elif zip_files:
    zip_path = zip_files[0]
    print(f"📦 Tìm thấy: {zip_path.name}")
    print(f"📂 Giải nén vào: {data_dir}")
    data_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(zip_path), "-d", str(data_dir)], check=True)
    # Nếu giải nén ra thêm 1 lớp thư mục con, dịch lên
    subdirs = [d for d in data_dir.iterdir() if d.is_dir() and d.name not in ["train","validation","test"]]
    if subdirs and not (data_dir / "train").exists():
        inner = subdirs[0]
        for d in inner.iterdir():
            d.rename(data_dir / d.name)
        inner.rmdir()
    print("✅ Giải nén xong!")
else:
    print("❌ Không tìm thấy file .zip trong:", REPO_DIR)
    print("   Hãy copy file data.zip vào thư mục:", REPO_DIR)

# Kiểm tra
def count_h5(path):
    return len(list(path.glob("*.h5"))) if path.exists() else 0

print(f"\n📊 Data location: {data_dir}")
print(f"   └─ train:      {count_h5(data_dir/'train')} files")
print(f"   └─ validation: {count_h5(data_dir/'validation')} files")
print(f"   └─ test:       {count_h5(data_dir/'test')} files")

## 🔧 **BƯỚC 2: Kiểm tra GPU**

In [ ]:
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"Số GPU:          {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory/1e9:.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError("GPU không có! Kiểm tra CUDA driver.")

print(f"\n✅ GPU sẵn sàng!")
print(f"{'='*60}")

## 📚 **BƯỚC 3: Install Dependencies**

In [ ]:
import sys

# Cài challenge package
print("📦 Cài đặt challenge package...")
%pip install -e {REPO_DIR} -q

# Cài model dependencies
print("📦 Cài đặt model dependencies...")
%pip install -r {REPO_DIR}/models_implementation/requirements.txt -q
%pip install tensorboard jupyter -q

print("\n✅ Tất cả dependencies đã được cài đặt!")

## ⚙️ **BƯỚC 4: Cấu hình Training**

Chọn config phù hợp:

| Config | Epochs | Thời gian (2x RTX 2080) | V-measure |
|--------|--------|-------------------------|-----------|
| quick    | 3  | ~1-2 giờ  | ~0.75 |
| standard | 8  | ~4-6 giờ  | ~0.88 |
| extended | 12 | ~8-10 giờ | ~0.89 |

In [ ]:
import os
from pathlib import Path

# ============== CHỌN CONFIG ==============
TRAINING_CONFIG = "standard"  # Options: "quick", "standard", "extended"
GPU_IDS = "0,1"               # "0" = chỉ GPU 0 | "1" = chỉ GPU 1 | "0,1" = cả 2
# =========================================

configs = {
    "quick":    {"num_epochs": 3,  "validate_every": 1, "estimated_hours": "1-2"},
    "standard": {"num_epochs": 8,  "validate_every": 2, "estimated_hours": "4-6"},
    "extended": {"num_epochs": 12, "validate_every": 2, "estimated_hours": "8-10"},
}
config = configs[TRAINING_CONFIG]

output_dir = REPO_DIR / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Số worker tối ưu cho server
num_gpus   = len(GPU_IDS.split(","))
num_workers = 4 * num_gpus

print(f"{'='*60}")
print(f"📋 Training Configuration: {TRAINING_CONFIG.upper()}")
print(f"{'='*60}")
print(f"GPU:              {GPU_IDS} ({num_gpus} GPU)")
print(f"Epochs:           {config['num_epochs']}")
print(f"Batch size:       8  (4 per GPU khi dùng 2 GPU)")
print(f"Validate every:   {config['validate_every']} epochs")
print(f"Num workers:      {num_workers}")
print(f"Data:             {data_dir}")
print(f"Output:           {output_dir}")
print(f"Thời gian dự kiến: {config['estimated_hours']} giờ")
print(f"{'='*60}")

## 🧪 **BƯỚC 5: Demo Training (subset 100k windows)** — *Tùy chọn*

Chạy nhanh để kiểm tra pipeline hoạt động đúng. Bỏ qua nếu muốn vào thẳng full training.

In [ ]:
import time, os

demo_output_dir = REPO_DIR / "outputs_demo_subset"

print("="*60)
print("🧪 DEMO TRAINING (subset 100k windows)")
print("="*60)

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {demo_output_dir} \
    --subset_size 100000 \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers}

elapsed = (time.time() - start_time) / 3600
print("\n" + "="*60)
print("✅ DEMO TRAINING HOÀN THÀNH!")
print(f"⏱️  Thời gian: {elapsed:.2f} giờ")
print(f"💾 Checkpoints: {demo_output_dir}")
print("="*60)

## 🚀 **BƯỚC 6: BẮT ĐẦU FULL TRAINING**

⏰ **Thời gian:** 4-6 giờ (standard config trên 2x RTX 2080)

💡 **Tips:**
- Checkpoints tự động lưu mỗi epoch
- Nếu bị ngắt giữa chừng, dùng `--resume` để tiếp tục
- Xem log realtime: mở terminal chạy `tail -f training.log`

In [ ]:
import time, os

print("\n" + "="*60)
print("🚀 BẮT ĐẦU FULL TRAINING")
print("="*60)
print(f"Data:    {data_dir}")
print(f"Output:  {output_dir}")
print(f"Epochs:  {config['num_epochs']}")
print(f"GPU:     {GPU_IDS}")
print(f"Dự kiến: {config['estimated_hours']} giờ")
print("="*60 + "\n")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers}

elapsed = (time.time() - start_time) / 3600

print("\n" + "="*60)
print("🎉 TRAINING HOÀN THÀNH!")
print("="*60)
print(f"⏱️  Thời gian thực tế: {elapsed:.2f} giờ")
print(f"💾 Checkpoints: {output_dir}")
print("="*60)

## 🔄 **TIẾP TỤC TRAINING (nếu bị ngắt giữa chừng)**

In [ ]:
import glob, os

# Tìm checkpoint mới nhất
checkpoints = sorted(glob.glob(str(output_dir / "run_*/checkpoint_epoch_*.pt")))

if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"✅ Checkpoint mới nhất: {latest_ckpt}")
    print("\nChạy cell dưới để tiếp tục:")
else:
    print("⚠️  Không tìm thấy checkpoint nào")
    latest_ckpt = ""

In [ ]:
import time, os

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS
start_time = time.time()

!python {REPO_DIR}/models_implementation/train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size 8 \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length 1000 \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers {num_workers} \
    --resume {latest_ckpt}

elapsed = (time.time() - start_time) / 3600
print(f"\n⏱️  Thời gian: {elapsed:.2f} giờ")

## 📊 **BƯỚC 7: Tìm Best Model**

In [ ]:
import glob, os

runs = sorted(glob.glob(str(output_dir / "run_*")))

if runs:
    latest_run = runs[-1]
    best_model = os.path.join(latest_run, "best_model.pt")

    print(f"{'='*60}")
    print("🏆 BEST MODEL")
    print(f"{'='*60}")

    if os.path.exists(best_model):
        print(f"✅ Model path: {best_model}")
        print(f"📁 Run dir:    {latest_run}")

        # Đọc config đã lưu
        import json
        config_file = os.path.join(latest_run, "config.json")
        if os.path.exists(config_file):
            with open(config_file) as f:
                saved_config = json.load(f)
            print("\n📋 Config đã dùng:")
            print(json.dumps(saved_config, indent=2))
    else:
        print("⚠️  best_model.pt chưa được tạo")
else:
    print("⚠️  Không tìm thấy training runs")

print(f"{'='*60}")

## 📈 **BƯỚC 8: Đánh giá Model**

Evaluate trên validation set.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS

if os.path.exists(best_model):
    print("📊 Đánh giá model trên validation set...\n")

    !python {REPO_DIR}/models_implementation/inference.py \
        --checkpoint {best_model} \
        --data_dir {data_dir} \
        --subset validation \
        --batch_size 8 \
        --save_results {latest_run}/validation_results.json

    print(f"\n✅ Kết quả đã lưu tại: {latest_run}/validation_results.json")
else:
    print("⚠️  Best model không tồn tại. Chạy training trước.")

## 📊 **BONUS: TensorBoard**

In [ ]:
%load_ext tensorboard

tb_logs = sorted(glob.glob(str(output_dir / "run_*/tensorboard")))

if tb_logs:
    latest_tb = tb_logs[-1]
    print(f"📊 TensorBoard logs: {latest_tb}")
    %tensorboard --logdir {latest_tb}
else:
    print("⚠️  Không tìm thấy TensorBoard logs")

---

## 🎯 **KẾT QUẢ MONG ĐỢI (2x RTX 2080)**

| Config | Epochs | Thời gian | V-measure |
|--------|--------|-----------|-----------|
| quick    | 3  | ~1-2 giờ  | ~0.75 |
| standard | 8  | ~4-6 giờ  | ~0.88 |
| extended | 12 | ~8-10 giờ | ~0.89 |

---

## 💡 **TROUBLESHOOTING**

### ❌ "CUDA Out of Memory"
```python
# Giảm batch_size trong BƯỚC 4 hoặc thêm vào lệnh train:
--batch_size 4
```

### ❌ Bị ngắt giữa chừng
Dùng cell **TIẾP TỤC TRAINING** ở trên, checkpoint đã lưu tự động.

### ❌ Muốn chạy nền (không cần giữ Jupyter mở)
```bash
# Trong terminal trên server:
nohup python train.py --data_dir ... --output_dir ... > training.log 2>&1 &
tail -f training.log
```